# 03 - PostgreSQL Database Loading  
  
*AFL Matchday Demand Forecasting - Loading and Verifying the Prepared Data*

## 1. Objective and Scope

### 1.1 Notebook Objective

This notebook loads the database-ready tables produced in Notebook 02 into PostgreSQL and verifies that the stored data is complete and consistent with the exported source files.

The notebook runs separate SQL scripts to create the required schemas and tables, loads the prepared CSV files, and performs focused checks on row counts, keys, critical missing values, and table relationships.

Data cleaning and standardisation are not repeated here. Feature engineering, dataset splitting, and model development are reserved for later stages of the project.

### 1.2 Input Files and Expected Database Tables

The notebook uses the five validated CSV files exported by Notebook 02 from:

`data/interim/postgres_ready/`

These files will be loaded into the PostgreSQL `staging` schema. The `staging` schema stores cleaned and standardised project data before feature engineering begins.

| Input CSV | Target PostgreSQL Table | Current Rows | Record Grain | Key |
|---|---|---:|---|---|
| `historical_matches.csv` | `staging.historical_matches` | 2,879 | One completed historical match | `match_id` |
| `squiggle_matches_2026_snapshot.csv` | `staging.squiggle_matches_2026_snapshot` | 218 | One match record per API snapshot | `snapshot_date`, `source_game_id` |
| `team_reference.csv` | `staging.team_reference` | 18 | One standardised AFL team | `team_id` |
| `venue_reference.csv` | `staging.venue_reference` | 27 | One standardised venue | `venue_name` |
| `school_holidays_daily.csv` | `staging.school_holidays_daily` | 67,208 | One capital city and calendar date | `city`, `calendar_date` |

The expected row counts provide reference values for the database-loading checks later in this notebook.

### 1.3 Processing Boundaries

This notebook is limited to the initial PostgreSQL loading and verification of the five prepared project tables.

#### Included

- Read the five validated CSV files from `data/interim/postgres_ready/`.
- Connect to PostgreSQL using settings stored in the local `.env` file.
- Run the separate SQL scripts that create the required schemas and tables.
- Perform a controlled full load of the five prepared tables.
- Verify row counts, keys, critical missing values, and selected table relationships.
- Leave the database ready for the feature-engineering stage.

#### Processing Rules

- The input CSV files are treated as read-only and are not modified.
- Database credentials are not displayed in notebook outputs or committed to GitHub.
- Schema and table definitions remain in separate files under `sql/`.
- The notebook runs the SQL files but does not duplicate their full SQL code.
- The v1 workflow uses a full table load rather than incremental updates.

#### Not Included

- Re-cleaning or correcting the Notebook 02 outputs.
- Loading the original raw CSV, JSON, RDA, or source-document files.
- Feature engineering or analytical dataset construction.
- Train, validation, and test dataset splitting.
- Model training, evaluation, or prediction.
- Power BI reporting or AWS automation.

## 2. Set up the Loading Environment

### 2.1 Import Libraries and Resolve Project Paths

In [1]:
import os
from pathlib import Path

import pandas as pd
import psycopg
from dotenv import load_dotenv


# Resolve the project root from either the repository root or notebooks directory.
current_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    current_directory.parent
    if current_directory.name.lower() == "notebooks"
    else current_directory
)

# Define the input, SQL, and local environment paths used by this notebook.
POSTGRES_READY_DIR = PROJECT_ROOT / "data" / "interim" / "postgres_ready"
SQL_DIR = PROJECT_ROOT / "sql"
ENV_FILE = PROJECT_ROOT / ".env"

# Summarise the required project paths before starting database operations.
path_summary = pd.DataFrame(
    [
        {
            "resource": "Project root",
            "path": str(PROJECT_ROOT),
            "expected_type": "directory",
            "exists": PROJECT_ROOT.is_dir(),
        },
        {
            "resource": "Database-ready data",
            "path": str(POSTGRES_READY_DIR),
            "expected_type": "directory",
            "exists": POSTGRES_READY_DIR.is_dir(),
        },
        {
            "resource": "SQL scripts",
            "path": str(SQL_DIR),
            "expected_type": "directory",
            "exists": SQL_DIR.is_dir(),
        },
        {
            "resource": "Environment file",
            "path": str(ENV_FILE),
            "expected_type": "file",
            "exists": ENV_FILE.is_file(),
        },
    ]
)

display(path_summary)

# Stop early if a required path is unavailable.
missing_resources = path_summary.loc[
    ~path_summary["exists"], "resource"
].tolist()

if missing_resources:
    raise FileNotFoundError(
        "Required project resources were not found: "
        + ", ".join(missing_resources)
    )

print(f"Project root: {PROJECT_ROOT}")
print("Project path check: PASSED")

,resource,path,expected_type,exists
0,Project root,D:\Projects\AFL_Project,directory,True
1,Database-ready data,D:\Projects\AFL_Project\data\interim\postgres_...,directory,True
2,SQL scripts,D:\Projects\AFL_Project\sql,directory,True
3,Environment file,D:\Projects\AFL_Project\.env,file,True


Project root: D:\Projects\AFL_Project
Project path check: PASSED


### 2.2 Load Database Connection Settings

The PostgreSQL connection settings are loaded from the local `.env` file.

The notebook validates that all required settings are available without displaying the database password. The resulting configuration will be used to open the PostgreSQL connection in the next section.

In [2]:
# Load the local database settings without replacing existing environment values.
load_dotenv(dotenv_path=ENV_FILE, override=False)

required_database_variables = [
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD",
]

# Confirm that every required connection setting contains a value.
missing_database_variables = [
    variable
    for variable in required_database_variables
    if not os.getenv(variable)
]

if missing_database_variables:
    raise EnvironmentError(
        "Required database settings are missing: "
        + ", ".join(missing_database_variables)
    )

# Build the connection configuration expected by psycopg.
DB_CONFIG = {
    "host": os.environ["DB_HOST"],
    "port": int(os.environ["DB_PORT"]),
    "dbname": os.environ["DB_NAME"],
    "user": os.environ["DB_USER"],
    "password": os.environ["DB_PASSWORD"],
}

# Display only non-sensitive connection information.
connection_settings_summary = pd.DataFrame(
    [
        {"setting": "Host", "value": DB_CONFIG["host"]},
        {"setting": "Port", "value": DB_CONFIG["port"]},
        {"setting": "Database", "value": DB_CONFIG["dbname"]},
        {"setting": "User", "value": DB_CONFIG["user"]},
        {"setting": "Password", "value": "Configured — value hidden"},
    ]
)

display(connection_settings_summary)

print("Database connection settings check: PASSED")

,setting,value
0,Host,localhost
1,Port,5432
2,Database,afl_project
3,User,afl_app
4,Password,Configured — value hidden


Database connection settings check: PASSED


### 2.3 Connect to PostgreSQL

This section opens a connection to PostgreSQL using the validated settings loaded from the local `.env` file.

A small system query confirms the active database, database user, and PostgreSQL server version. No project tables are created or modified at this stage.

In [3]:
# Open a reusable PostgreSQL connection for the following database operations.
connection = psycopg.connect(**DB_CONFIG)
connection.autocommit = False

# Confirm the active database context without exposing sensitive credentials.
with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT
            current_database(),
            current_user,
            current_setting('server_version');
        """
    )
    connection_details = cursor.fetchone()

connection_summary = pd.DataFrame(
    [connection_details],
    columns=[
        "database_name",
        "database_user",
        "postgresql_version",
    ],
)

display(connection_summary)

# Complete the read-only connection test and return to a clean transaction state.
connection.commit()

print(f"Connection open: {not connection.closed}")
print("PostgreSQL connection check: PASSED")

,database_name,database_user,postgresql_version
0,afl_project,afl_app,18.3


Connection open: True
PostgreSQL connection check: PASSED


### 2.4 Confirm the Database-Ready Input Files

This section confirms that the five database-ready CSV files are available and that their row counts match the validated outputs from Notebook 02.

The check does not repeat the earlier data-cleaning and quality-validation workflow. Its purpose is to prevent missing, incomplete, or incorrect files from being loaded into PostgreSQL.

In [9]:
# Define the five prepared input files and their expected row counts.
INPUT_FILE_SPECS = [
    {
        "file_name": "historical_matches.csv",
        "target_table": "staging.historical_matches",
    },
    {
        "file_name": "squiggle_matches_2026_snapshot.csv",
        "target_table": "staging.squiggle_matches_2026_snapshot",
    },
    {
        "file_name": "team_reference.csv",
        "target_table": "staging.team_reference",
    },
    {
        "file_name": "venue_reference.csv",
        "target_table": "staging.venue_reference",
    },
    {
        "file_name": "school_holidays_daily.csv",
        "target_table": "staging.school_holidays_daily",
    },
]

# Confirm that each file exists and contains the expected number of records.
input_file_checks = []

for file_spec in INPUT_FILE_SPECS:
    file_path = POSTGRES_READY_DIR / file_spec["file_name"]
    file_exists = file_path.is_file()

    actual_rows = (
        len(pd.read_csv(file_path))
        if file_exists
        else pd.NA
    )

    status = (
        "PASSED"
        if file_exists and actual_rows > 0
        else "FAILED"
    )

    input_file_checks.append(
        {
            "file_name": file_spec["file_name"],
            "target_table": file_spec["target_table"],
            "file_exists": file_exists,
            "actual_rows": actual_rows,
            "status": status,
        }
    )

input_file_summary = pd.DataFrame(input_file_checks)

display(input_file_summary)

# Stop before database creation if an input file is missing or incomplete.
if not input_file_summary["status"].eq("PASSED").all():
    raise ValueError(
        "One or more database-ready input files failed validation."
    )

# Store the current CSV row counts for the later database-loading checks.
source_row_counts = (
    input_file_summary
    .set_index("file_name")["actual_rows"]
    .astype(int)
    .to_dict()
)
print("Database-ready input file check: PASSED")

,file_name,target_table,file_exists,actual_rows,status
0,historical_matches.csv,staging.historical_matches,True,2879,PASSED
1,squiggle_matches_2026_snapshot.csv,staging.squiggle_matches_2026_snapshot,True,218,PASSED
2,team_reference.csv,staging.team_reference,True,18,PASSED
3,venue_reference.csv,staging.venue_reference,True,27,PASSED
4,school_holidays_daily.csv,staging.school_holidays_daily,True,67208,PASSED


Database-ready input file check: PASSED


## 3. Create the Database Schemas and Tables

### 3.1 Review the SQL Script Execution Order

Database structures are defined in separate, version-controlled SQL files rather than being embedded directly in the notebook.

The scripts must run in numerical order because the PostgreSQL schemas must exist before the project tables can be created.

| Execution Order | SQL Script | Purpose |
|---:|---|---|
| 1 | `00_create_schemas.sql` | Create the `raw`, `staging`, and `analytics` schemas if they do not already exist |
| 2 | `01_create_tables.sql` | Create the five prepared-data tables in the `staging` schema |

The current v1 data is loaded only into `staging`. The `raw` schema is not used to store the original source files, while the `analytics` schema is reserved for later feature-engineering outputs.

### 3.2 Run the Schema Creation Script

This section reads and executes `00_create_schemas.sql` to create the PostgreSQL schemas required by the project.

The script uses `CREATE SCHEMA IF NOT EXISTS`, allowing it to be run again without replacing existing schemas. The transaction is committed only after the complete script executes successfully.

In [5]:
schema_script_path = SQL_DIR / "00_create_schemas.sql"

# Confirm that the schema creation script exists and contains SQL.
if not schema_script_path.is_file():
    raise FileNotFoundError(
        f"Schema creation script not found: {schema_script_path}"
    )

schema_sql = schema_script_path.read_text(encoding="utf-8").strip()

if not schema_sql:
    raise ValueError("The schema creation script is empty.")

# Execute the complete schema script as one database transaction.
try:
    with connection.cursor() as cursor:
        cursor.execute(schema_sql)

    connection.commit()

except psycopg.Error:
    connection.rollback()
    raise

print(f"Executed SQL script: {schema_script_path.name}")
print("PostgreSQL schema creation: PASSED")

Executed SQL script: 00_create_schemas.sql
PostgreSQL schema creation: PASSED


### 3.3 Run the Table Creation Script

This section reads and executes `01_create_tables.sql` to create the five prepared-data tables in the PostgreSQL `staging` schema.

The reference tables are created before the match tables so that the required foreign-key relationships can be established. The transaction is committed only after the complete script executes successfully.

In [6]:
table_script_path = SQL_DIR / "01_create_tables.sql"

# Confirm that the table creation script exists and contains SQL.
if not table_script_path.is_file():
    raise FileNotFoundError(
        f"Table creation script not found: {table_script_path}"
    )

table_sql = table_script_path.read_text(encoding="utf-8").strip()

if not table_sql:
    raise ValueError("The table creation script is empty.")

# Execute all table definitions as one database transaction.
try:
    with connection.cursor() as cursor:
        cursor.execute(table_sql)

    connection.commit()

except psycopg.Error:
    connection.rollback()
    raise

print(f"Executed SQL script: {table_script_path.name}")
print("PostgreSQL table creation: PASSED")

Executed SQL script: 01_create_tables.sql
PostgreSQL table creation: PASSED


### 3.4 Confirm the Created Schema and Tables

This section queries PostgreSQL system metadata to confirm that the expected schemas and staging tables were created successfully.

The check validates the database objects themselves rather than relying only on the absence of SQL execution errors. No project data is loaded at this stage.

In [7]:
expected_schemas = {
    "raw",
    "staging",
    "analytics",
}

expected_staging_tables = {
    "historical_matches",
    "squiggle_matches_2026_snapshot",
    "team_reference",
    "venue_reference",
    "school_holidays_daily",
}

# Read the schema and table names directly from PostgreSQL metadata.
with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name IN ('raw', 'staging', 'analytics');
        """
    )
    created_schemas = {row[0] for row in cursor.fetchall()}

    cursor.execute(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'staging'
          AND table_type = 'BASE TABLE';
        """
    )
    created_staging_tables = {row[0] for row in cursor.fetchall()}

connection.commit()

# Summarise whether every expected database object is present.
database_object_check = pd.DataFrame(
    [
        {
            "object_type": "schema",
            "object_name": schema_name,
            "status": (
                "PASSED"
                if schema_name in created_schemas
                else "FAILED"
            ),
        }
        for schema_name in sorted(expected_schemas)
    ]
    + [
        {
            "object_type": "staging table",
            "object_name": table_name,
            "status": (
                "PASSED"
                if table_name in created_staging_tables
                else "FAILED"
            ),
        }
        for table_name in sorted(expected_staging_tables)
    ]
)

display(database_object_check)

if not database_object_check["status"].eq("PASSED").all():
    raise RuntimeError(
        "One or more required PostgreSQL objects were not created."
    )

print("PostgreSQL schema and table check: PASSED")

,object_type,object_name,status
0,schema,analytics,PASSED
1,schema,raw,PASSED
2,schema,staging,PASSED
3,staging table,historical_matches,PASSED
4,staging table,school_holidays_daily,PASSED
5,staging table,squiggle_matches_2026_snapshot,PASSED
6,staging table,team_reference,PASSED
7,staging table,venue_reference,PASSED


PostgreSQL schema and table check: PASSED


## 4. Load the Prepared Tables

### 4.1 Define the Table Loading Orders

The five prepared CSV files are loaded in an order that respects the foreign-key relationships defined in PostgreSQL.

| Loading Order | Target Table | Reason |
|---:|---|---|
| 1 | `staging.team_reference` | Team records must exist before match records can reference `team_id` |
| 2 | `staging.venue_reference` | Venue records must exist before match records can reference `venue_name` |
| 3 | `staging.historical_matches` | Historical matches depend on the team and venue reference tables |
| 4 | `staging.squiggle_matches_2026_snapshot` | Snapshot records use the same team and venue references |
| 5 | `staging.school_holidays_daily` | The holiday table has no foreign-key dependency and is loaded last |

The v1 workflow uses a controlled full load. Existing rows in the five staging tables are cleared before loading, while the table structures remain unchanged.

All five files are loaded within one database transaction. The transaction is committed only after every load succeeds; otherwise, the complete loading operation can be rolled back.

### 4.2 Load the Team and Venue Reference Tables

The staging tables are first cleared to support a controlled full reload without creating duplicate records.

A reusable PostgreSQL `COPY` function then streams the prepared CSV files into their target tables. The team and venue reference tables are loaded first because the two match tables depend on their keys.

The transaction remains uncommitted until all five tables have been loaded successfully.

In [10]:
def copy_csv_to_table(file_name: str, target_table: str) -> None:
    """Stream one prepared CSV file into its PostgreSQL staging table."""

    csv_path = POSTGRES_READY_DIR / file_name

    if not csv_path.is_file():
        raise FileNotFoundError(f"Input CSV not found: {csv_path}")

    copy_query = (
        f"COPY {target_table} FROM STDIN "
        "WITH (FORMAT CSV, HEADER TRUE, NULL '')"
    )

    # Stream the CSV in blocks instead of constructing row-level INSERT statements.
    with csv_path.open("rb") as source_file:
        with connection.cursor() as cursor:
            with cursor.copy(copy_query) as copy:
                while data_block := source_file.read(1024 * 1024):
                    copy.write(data_block)


try:
    # Clear all target tables within the same uncommitted transaction.
    with connection.cursor() as cursor:
        cursor.execute(
            """
            TRUNCATE TABLE
                staging.historical_matches,
                staging.squiggle_matches_2026_snapshot,
                staging.school_holidays_daily,
                staging.venue_reference,
                staging.team_reference;
            """
        )

    # Load the reference tables before the dependent match tables.
    copy_csv_to_table(
        "team_reference.csv",
        "staging.team_reference",
    )

    copy_csv_to_table(
        "venue_reference.csv",
        "staging.venue_reference",
    )

    # Confirm the number of reference records loaded in the current transaction.
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT 'team_reference', COUNT(*)
            FROM staging.team_reference

            UNION ALL

            SELECT 'venue_reference', COUNT(*)
            FROM staging.venue_reference;
            """
        )
        reference_counts = cursor.fetchall()

except (OSError, psycopg.Error):
    connection.rollback()
    raise


source_reference_rows = {
    "team_reference": source_row_counts["team_reference.csv"],
    "venue_reference": source_row_counts["venue_reference.csv"],
}

reference_load_summary = pd.DataFrame(
    reference_counts,
    columns=["table_name", "loaded_rows"],
)

reference_load_summary["source_csv_rows"] = (
    reference_load_summary["table_name"].map(source_reference_rows)
)

reference_load_summary["status"] = (
    reference_load_summary["loaded_rows"]
    == reference_load_summary["source_csv_rows"]
).map({True: "PASSED", False: "FAILED"})

display(reference_load_summary)

if not reference_load_summary["status"].eq("PASSED").all():
    connection.rollback()
    raise ValueError("Reference-table row-count validation failed.")

print("Reference table loading: PASSED — transaction not yet committed")

,table_name,loaded_rows,source_csv_rows,status
0,team_reference,18,18,PASSED
1,venue_reference,27,27,PASSED


Reference table loading: PASSED — transaction not yet committed


### 4.3 Load the Historical Match Table

This section loads the prepared historical match records after the team and venue reference tables are available.

PostgreSQL applies the primary-key and foreign-key constraints during loading. The loaded row count is checked against the 2,879 records exported by Notebook 02, while the transaction remains uncommitted.

In [11]:
source_historical_rows = source_row_counts["historical_matches.csv"]

try:
    # Load the prepared historical match records.
    copy_csv_to_table(
        "historical_matches.csv",
        "staging.historical_matches",
    )

    # Confirm the number of historical matches loaded in the current transaction.
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT COUNT(*) FROM staging.historical_matches;"
        )
        loaded_historical_rows = cursor.fetchone()[0]

    if loaded_historical_rows != source_historical_rows:
        raise ValueError(
            "The PostgreSQL row count does not match "
            "the historical match CSV."
        )

except (OSError, psycopg.Error, ValueError):
    connection.rollback()
    raise

print(f"Historical match rows loaded: {loaded_historical_rows:,}")
print("Historical match loading: PASSED — transaction not yet committed")

Historical match rows loaded: 2,879
Historical match loading: PASSED — transaction not yet committed


### 4.4 Load the 2026 Squiggle Match Snapshot

This section loads the prepared 2026 Squiggle match snapshot into PostgreSQL.

Each record is identified by the combination of `snapshot_date` and `source_game_id`. Completed matches, future fixtures, and finals placeholder records are retained in the same snapshot table.

The loaded row count is compared with the current CSV rather than with a permanently fixed value. The transaction remains uncommitted.

In [12]:
source_snapshot_rows = source_row_counts[
    "squiggle_matches_2026_snapshot.csv"
]

try:
    # Load the prepared Squiggle match snapshot.
    copy_csv_to_table(
        "squiggle_matches_2026_snapshot.csv",
        "staging.squiggle_matches_2026_snapshot",
    )

    # Confirm the number of snapshot records loaded in the current transaction.
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM staging.squiggle_matches_2026_snapshot;
            """
        )
        loaded_snapshot_rows = cursor.fetchone()[0]

    if loaded_snapshot_rows != source_snapshot_rows:
        raise ValueError(
            "The PostgreSQL row count does not match "
            "the Squiggle snapshot CSV."
        )

except (OSError, psycopg.Error, ValueError):
    connection.rollback()
    raise

print(f"Squiggle snapshot rows loaded: {loaded_snapshot_rows:,}")
print("Squiggle snapshot loading: PASSED — transaction not yet committed")

Squiggle snapshot rows loaded: 218
Squiggle snapshot loading: PASSED — transaction not yet committed


### 4.5 Load the Daily School-Holiday Table

This section loads the combined daily school-holiday records covering the eight Australian capital cities.

Each instance represents one city and calendar date. PostgreSQL enforces the composite primary key of `city` and `calendar_date`, while the loaded row count is compared with the current CSV.

This is the final table loaded before the transaction is committed.

In [13]:
source_holiday_rows = source_row_counts[
    "school_holidays_daily.csv"
]

try:
    # Load the prepared daily school-holiday records.
    copy_csv_to_table(
        "school_holidays_daily.csv",
        "staging.school_holidays_daily",
    )

    # Confirm the number of holiday records loaded in the current transaction.
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM staging.school_holidays_daily;
            """
        )
        loaded_holiday_rows = cursor.fetchone()[0]

    if loaded_holiday_rows != source_holiday_rows:
        raise ValueError(
            "The PostgreSQL row count does not match "
            "the daily school-holiday CSV."
        )

except (OSError, psycopg.Error, ValueError):
    connection.rollback()
    raise

print(f"School-holiday rows loaded: {loaded_holiday_rows:,}")
print("School-holiday loading: PASSED — transaction not yet committed")

School-holiday rows loaded: 67,208
School-holiday loading: PASSED — transaction not yet committed


### 4.6 Commit the Database Transaction

All five prepared tables have now been loaded and matched to their current CSV row counts.

This section commits the complete full-load transaction, making the refreshed staging data permanent in PostgreSQL. The database connection remains open for the validation queries in Section 5.

In [14]:
# Commit the full five-table load only after every loading check has passed.
try:
    connection.commit()

except psycopg.Error:
    connection.rollback()
    raise

print("Five-table staging load committed successfully.")
print("PostgreSQL data-loading transaction: PASSED")

Five-table staging load committed successfully.
PostgreSQL data-loading transaction: PASSED


## 5. Validate the Loaded Database

### 5.1 Compare PostgreSQL and CSV Row Counts

This section performs the first post-commit validation by comparing the number of records stored in each PostgreSQL table with the number of records in its current source CSV.

The comparison confirms that the committed database load is complete without relying on permanently fixed row counts.

In [15]:
# Map each PostgreSQL table to the current row count of its source CSV.
table_source_rows = {
    "historical_matches": source_row_counts[
        "historical_matches.csv"
    ],
    "squiggle_matches_2026_snapshot": source_row_counts[
        "squiggle_matches_2026_snapshot.csv"
    ],
    "team_reference": source_row_counts[
        "team_reference.csv"
    ],
    "venue_reference": source_row_counts[
        "venue_reference.csv"
    ],
    "school_holidays_daily": source_row_counts[
        "school_holidays_daily.csv"
    ],
}

# Read the committed row counts from PostgreSQL.
with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT 'historical_matches', COUNT(*)
        FROM staging.historical_matches

        UNION ALL

        SELECT 'squiggle_matches_2026_snapshot', COUNT(*)
        FROM staging.squiggle_matches_2026_snapshot

        UNION ALL

        SELECT 'team_reference', COUNT(*)
        FROM staging.team_reference

        UNION ALL

        SELECT 'venue_reference', COUNT(*)
        FROM staging.venue_reference

        UNION ALL

        SELECT 'school_holidays_daily', COUNT(*)
        FROM staging.school_holidays_daily;
        """
    )
    database_row_counts = cursor.fetchall()

connection.commit()

row_count_validation = pd.DataFrame(
    database_row_counts,
    columns=["table_name", "postgresql_rows"],
)

row_count_validation["source_csv_rows"] = (
    row_count_validation["table_name"].map(table_source_rows)
)

row_count_validation["status"] = (
    row_count_validation["postgresql_rows"]
    == row_count_validation["source_csv_rows"]
).map({True: "PASSED", False: "FAILED"})

display(row_count_validation)

if not row_count_validation["status"].eq("PASSED").all():
    raise ValueError(
        "One or more PostgreSQL row counts do not match their source CSV."
    )

print("PostgreSQL-to-CSV row-count validation: PASSED")

,table_name,postgresql_rows,source_csv_rows,status
0,historical_matches,2879,2879,PASSED
1,squiggle_matches_2026_snapshot,218,218,PASSED
2,team_reference,18,18,PASSED
3,venue_reference,27,27,PASSED
4,school_holidays_daily,67208,67208,PASSED


PostgreSQL-to-CSV row-count validation: PASSED


### 5.2 Validate Primary and Composite Keys

PostgreSQL primary keys enforce both uniqueness and non-null values during data loading. Therefore, this section does not repeat the duplicate-record checks already completed in Notebook 02.

Instead, PostgreSQL metadata is queried to confirm that each staging table has the intended single-column or composite primary key.

In [16]:
expected_primary_keys = {
    "historical_matches": (
        "match_id",
    ),
    "squiggle_matches_2026_snapshot": (
        "snapshot_date",
        "source_game_id",
    ),
    "team_reference": (
        "team_id",
    ),
    "venue_reference": (
        "venue_name",
    ),
    "school_holidays_daily": (
        "city",
        "calendar_date",
    ),
}

# Read the primary-key columns directly from PostgreSQL metadata.
with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT
            tc.table_name,
            kcu.column_name,
            kcu.ordinal_position
        FROM information_schema.table_constraints AS tc
        INNER JOIN information_schema.key_column_usage AS kcu
            ON tc.constraint_catalog = kcu.constraint_catalog
           AND tc.constraint_schema = kcu.constraint_schema
           AND tc.constraint_name = kcu.constraint_name
        WHERE tc.table_schema = 'staging'
          AND tc.constraint_type = 'PRIMARY KEY'
        ORDER BY
            tc.table_name,
            kcu.ordinal_position;
        """
    )
    primary_key_rows = cursor.fetchall()

connection.commit()

# Group the returned columns by table while preserving composite-key order.
postgresql_primary_keys = {}

for table_name, column_name, _ in primary_key_rows:
    postgresql_primary_keys.setdefault(table_name, []).append(column_name)

primary_key_validation = pd.DataFrame(
    [
        {
            "table_name": table_name,
            "expected_key": " + ".join(expected_columns),
            "postgresql_key": " + ".join(
                postgresql_primary_keys.get(table_name, [])
            ),
            "status": (
                "PASSED"
                if tuple(postgresql_primary_keys.get(table_name, []))
                == expected_columns
                else "FAILED"
            ),
        }
        for table_name, expected_columns in expected_primary_keys.items()
    ]
)

display(primary_key_validation)

if not primary_key_validation["status"].eq("PASSED").all():
    raise ValueError(
        "One or more PostgreSQL primary keys do not match the table design."
    )

print("PostgreSQL primary-key validation: PASSED")

,table_name,expected_key,postgresql_key,status
0,historical_matches,match_id,match_id,PASSED
1,squiggle_matches_2026_snapshot,snapshot_date + source_game_id,snapshot_date + source_game_id,PASSED
2,team_reference,team_id,team_id,PASSED
3,venue_reference,venue_name,venue_name,PASSED
4,school_holidays_daily,city + calendar_date,city + calendar_date,PASSED


PostgreSQL primary-key validation: PASSED


### 5.3 Validate Missing Critical Values

Most required fields are already protected by PostgreSQL primary-key and `NOT NULL` constraints.

This section therefore focuses only on conditionally required Squiggle fields. Finals placeholder records may legitimately have missing match, team, and venue details, while non-placeholder matches must have a match identifier and standardised mappings. Completed matches must also have final scores.

In [17]:
# Check fields that are required only for specific Squiggle record types.
with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT
            'Non-placeholder records missing match_id',
            COUNT(*)
        FROM staging.squiggle_matches_2026_snapshot
        WHERE snapshot_record_type <> 'finals_placeholder'
          AND match_id IS NULL

        UNION ALL

        SELECT
            'Non-placeholder records missing team or venue mappings',
            COUNT(*)
        FROM staging.squiggle_matches_2026_snapshot
        WHERE snapshot_record_type <> 'finals_placeholder'
          AND (
              home_team_id IS NULL
              OR away_team_id IS NULL
              OR venue_name IS NULL
          )

        UNION ALL

        SELECT
            'Completed matches missing final scores',
            COUNT(*)
        FROM staging.squiggle_matches_2026_snapshot
        WHERE snapshot_record_type = 'completed_match'
          AND (
              home_score IS NULL
              OR away_score IS NULL
          );
        """
    )
    critical_missing_rows = cursor.fetchall()

connection.commit()

critical_missing_validation = pd.DataFrame(
    critical_missing_rows,
    columns=["check", "unexpected_missing_records"],
)

critical_missing_validation["status"] = (
    critical_missing_validation["unexpected_missing_records"]
    .eq(0)
    .map({True: "PASSED", False: "FAILED"})
)

display(critical_missing_validation)

if not critical_missing_validation["status"].eq("PASSED").all():
    raise ValueError(
        "Unexpected missing values were found in critical match fields."
    )

print("Conditional critical-value validation: PASSED")

,check,unexpected_missing_records,status
0,Non-placeholder records missing match_id,0,PASSED
1,Non-placeholder records missing team or venue ...,0,PASSED
2,Completed matches missing final scores,0,PASSED


Conditional critical-value validation: PASSED


### 5.4 Validate Team, Venue, and Calendar Relationships

This section confirms that match records can be joined to their required team and venue reference records.

It also verifies that matches with an assigned Australian school-holiday city can be joined to the daily holiday table using `school_holiday_city` and `match_date`.

International matches with no applicable Australian school-holiday city are intentionally excluded from the calendar checks.

In [18]:
# Count unmatched team, venue, and school-calendar relationships.
with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT
            'Historical team references',
            COUNT(*)
        FROM staging.historical_matches AS matches
        LEFT JOIN staging.team_reference AS home_teams
            ON matches.home_team_id = home_teams.team_id
        LEFT JOIN staging.team_reference AS away_teams
            ON matches.away_team_id = away_teams.team_id
        WHERE home_teams.team_id IS NULL
           OR away_teams.team_id IS NULL

        UNION ALL

        SELECT
            'Historical venue references',
            COUNT(*)
        FROM staging.historical_matches AS matches
        LEFT JOIN staging.venue_reference AS venues
            ON matches.venue_name = venues.venue_name
        WHERE venues.venue_name IS NULL

        UNION ALL

        SELECT
            'Squiggle team and venue references',
            COUNT(*)
        FROM staging.squiggle_matches_2026_snapshot AS matches
        LEFT JOIN staging.team_reference AS home_teams
            ON matches.home_team_id = home_teams.team_id
        LEFT JOIN staging.team_reference AS away_teams
            ON matches.away_team_id = away_teams.team_id
        LEFT JOIN staging.venue_reference AS venues
            ON matches.venue_name = venues.venue_name
        WHERE (
                matches.home_team_id IS NOT NULL
                AND home_teams.team_id IS NULL
              )
           OR (
                matches.away_team_id IS NOT NULL
                AND away_teams.team_id IS NULL
              )
           OR (
                matches.venue_name IS NOT NULL
                AND venues.venue_name IS NULL
              )

        UNION ALL

        SELECT
            'Historical school-calendar coverage',
            COUNT(*)
        FROM staging.historical_matches AS matches
        LEFT JOIN staging.school_holidays_daily AS holidays
            ON matches.school_holiday_city = holidays.city
           AND matches.match_date = holidays.calendar_date
        WHERE matches.school_holiday_city IS NOT NULL
          AND holidays.city IS NULL

        UNION ALL

        SELECT
            'Squiggle school-calendar coverage',
            COUNT(*)
        FROM staging.squiggle_matches_2026_snapshot AS matches
        LEFT JOIN staging.school_holidays_daily AS holidays
            ON matches.school_holiday_city = holidays.city
           AND matches.match_date = holidays.calendar_date
        WHERE matches.school_holiday_city IS NOT NULL
          AND holidays.city IS NULL;
        """
    )
    relationship_rows = cursor.fetchall()

connection.commit()

relationship_validation = pd.DataFrame(
    relationship_rows,
    columns=["check", "unmatched_records"],
)

relationship_validation["status"] = (
    relationship_validation["unmatched_records"]
    .eq(0)
    .map({True: "PASSED", False: "FAILED"})
)

display(relationship_validation)

if not relationship_validation["status"].eq("PASSED").all():
    raise ValueError(
        "One or more PostgreSQL table relationships failed validation."
    )

print("PostgreSQL relationship validation: PASSED")

,check,unmatched_records,status
0,Squiggle team and venue references,0,PASSED
1,Historical venue references,0,PASSED
2,Historical team references,0,PASSED
3,Squiggle school-calendar coverage,0,PASSED
4,Historical school-calendar coverage,0,PASSED


PostgreSQL relationship validation: PASSED


### 5.5 Run Simple Sample Queries

This section runs two small SQL queries to demonstrate that the loaded tables can support the next analytical stage.

The first query retrieves recent historical matches with attendance and school-holiday information. The second retrieves future fixtures with the same calendar context.

These queries confirm database usability without performing feature engineering or model analysis.

In [19]:
with connection.cursor() as cursor:
    # Retrieve a small sample of recent historical matches.
    cursor.execute(
        """
        SELECT
            matches.match_id,
            matches.match_date,
            matches.home_team,
            matches.away_team,
            matches.venue_name,
            matches.attendance,
            holidays.is_school_holiday
        FROM staging.historical_matches AS matches
        LEFT JOIN staging.school_holidays_daily AS holidays
            ON matches.school_holiday_city = holidays.city
           AND matches.match_date = holidays.calendar_date
        ORDER BY
            matches.match_date DESC,
            matches.match_id
        LIMIT 5;
        """
    )

    historical_sample = pd.DataFrame(
        cursor.fetchall(),
        columns=[
            "match_id",
            "match_date",
            "home_team",
            "away_team",
            "venue_name",
            "attendance",
            "is_school_holiday",
        ],
    )

    # Retrieve the future fixtures available in the current API snapshot.
    cursor.execute(
        """
        SELECT
            matches.snapshot_date,
            matches.source_game_id,
            matches.match_date,
            matches.start_time,
            matches.home_team,
            matches.away_team,
            matches.venue_name,
            holidays.is_school_holiday
        FROM staging.squiggle_matches_2026_snapshot AS matches
        LEFT JOIN staging.school_holidays_daily AS holidays
            ON matches.school_holiday_city = holidays.city
           AND matches.match_date = holidays.calendar_date
        WHERE matches.snapshot_record_type = 'future_fixture'
        ORDER BY
            matches.match_date,
            matches.start_time,
            matches.source_game_id;
        """
    )

    future_fixture_sample = pd.DataFrame(
        cursor.fetchall(),
        columns=[
            "snapshot_date",
            "source_game_id",
            "match_date",
            "start_time",
            "home_team",
            "away_team",
            "venue_name",
            "is_school_holiday",
        ],
    )

connection.commit()

print("Recent historical match sample:")
display(historical_sample)

print("Future fixture sample:")
display(future_fixture_sample)

print("PostgreSQL sample-query check: PASSED")

Recent historical match sample:


,match_id,match_date,home_team,away_team,venue_name,attendance,is_school_holiday
0,20250927_H07_A02,2025-09-27,Geelong,Brisbane Lions,MCG,100022,True
1,20250920_H04_A02,2025-09-20,Collingwood,Brisbane Lions,MCG,96023,True
2,20250919_H07_A10,2025-09-19,Geelong,Hawthorn,MCG,99567,False
3,20250913_H02_A08,2025-09-13,Brisbane Lions,Gold Coast,Gabba,36628,False
4,20250912_H01_A10,2025-09-12,Adelaide,Hawthorn,Adelaide Oval,52005,False


Future fixture sample:


,snapshot_date,source_game_id,match_date,start_time,home_team,away_team,venue_name,is_school_holiday
0,2026-08-17,38697,2026-08-20,19:30:00,St Kilda,Gold Coast,Docklands,False
1,2026-08-17,38692,2026-08-21,19:40:00,Collingwood,Brisbane Lions,MCG,False
2,2026-08-17,38693,2026-08-22,13:15:00,Carlton,Fremantle,Docklands,False
3,2026-08-17,38696,2026-08-22,16:15:00,Melbourne,Western Bulldogs,MCG,False
4,2026-08-17,38699,2026-08-22,19:40:00,Adelaide,Greater Western Sydney,Adelaide Oval,False
5,2026-08-17,38695,2026-08-22,19:45:00,Geelong,Richmond,Kardinia Park,False
6,2026-08-17,38694,2026-08-23,12:20:00,Essendon,Port Adelaide,Docklands,False
7,2026-08-17,38698,2026-08-23,15:20:00,Sydney,North Melbourne,SCG,False
8,2026-08-17,38700,2026-08-23,17:20:00,West Coast,Hawthorn,Perth Stadium,False


PostgreSQL sample-query check: PASSED


### 5.6 Summarise the Validation Results

All post-load validation checks passed successfully.

- PostgreSQL row counts matched the current source CSV files for all five staging tables.
- All single-column and composite primary keys matched the intended table design.
- No unexpected missing values were found in conditionally required Squiggle fields.
- Team, venue, and school-calendar relationships contained no unmatched records.
- Sample SQL queries successfully returned historical matches and future fixtures with school-holiday context.

The committed PostgreSQL staging data is complete, internally consistent, and ready for the feature-engineering stage.

## 6. Summarise and Confirm the Next Step

### 6.1 Summarise the Loaded Tables

The database-loading workflow was completed successfully. The five prepared CSV files were loaded into corresponding tables under the PostgreSQL `staging` schema.

| PostgreSQL Table | Record Grain | Purpose |
|---|---|---|
| `staging.team_reference` | One AFL team | Provides consistent team identifiers and names |
| `staging.venue_reference` | One standardised venue | Connects match venues with their corresponding cities |
| `staging.historical_matches` | One completed historical match | Supports attendance analysis and model development |
| `staging.squiggle_matches_2026_snapshot` | One match record per API snapshot | Provides completed matches, future fixtures, and finals placeholders |
| `staging.school_holidays_daily` | One capital city per calendar date | Provides daily school-holiday information for match dates |

The loading process used one database transaction. All five tables were committed only after their source row counts had been confirmed. The post-load checks also confirmed the intended primary keys, required values, reference relationships, and sample-query results.

### 6.2 Document Database Limitations

The current database implementation has the following scope and limitations:

- PostgreSQL is running in a local development environment rather than a production cloud environment.
- The five staging tables use a full-refresh loading method because the current datasets are small and the workflow prioritises simplicity and reproducibility.
- When the prepared CSV files are updated, Notebook 02 must be rerun before this loading notebook is executed again.
- The Squiggle data represents the API snapshot captured on 17 August 2026. Later fixture changes are not included in this snapshot.
- Eleven finals placeholder records are intentionally retained without assigned teams or `match_id` values because their matchups were not yet known.
- School-holiday records are limited to the eight Australian capital cities required by the project.
- Feature engineering, model-ready tables, model training, and prediction are outside the scope of this notebook.
- Database credentials are stored locally in `.env` and are not committed to the GitHub repository.

### 6.3 Confirm the Feature-Engineering Handoff

All database-loading and validation activities are now complete. The active PostgreSQL connection is closed explicitly so that the notebook finishes without leaving an unnecessary database session open.

In [20]:
# Close the PostgreSQL connection after completing all loading and validation work.
if not connection.closed:
    connection.close()

connection_status = "CLOSED" if connection.closed else "OPEN"
print(f"PostgreSQL connection status: {connection_status}")

PostgreSQL connection status: CLOSED


**Conclusion and Next Step**

This notebook created the required PostgreSQL structures, loaded the five prepared datasets, and validated the committed database records.

The next stage will be completed in:

`04_feature_engineering.ipynb`

The next notebook will query the PostgreSQL staging tables and create a model-ready match dataset. It will focus on:

- joining match, venue, and school-holiday information;
- defining the attendance prediction target;
- creating match, calendar, venue, and historical performance features;
- ensuring that every feature uses only information available before each match;
- assigning records to appropriate modelling periods; and
- validating the final model-ready dataset before model training.

Model training and model comparison will be completed in a later notebook.